# Hierarchical Models: Complete vs No vs Partial Pooling

## Learning Objectives

By completing this notebook, you will:
1. Understand the pooling spectrum: complete, no pooling, and partial pooling
2. Recognize when hierarchical models provide the most benefit
3. Implement all three approaches for commodity price forecasting
4. Quantify the bias-variance tradeoff across pooling strategies
5. Apply shrinkage concepts to improve estimates for small-sample commodities
6. Build PyMC hierarchical models for multi-commodity forecasting

## Prerequisites
- Bayesian regression fundamentals
- Understanding of variance and bias
- PyMC model building experience

## Estimated Time: 90 minutes

---

In [ ]:
learning_objectives(["Understand the pooling spectrum: complete, no pooling, and partial pooling", "Recognize when hierarchical models provide the most benefit", "Implement all three approaches for commodity price forecasting", "Quantify the bias-variance tradeoff across pooling strategies", "Apply shrinkage concepts to improve estimates for small-sample commodities", "Build PyMC hierarchical models for multi-commodity forecasting"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()

## 1. Setup and Imports

In [ ]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Bayesian stack
import pymc as pm
import arviz as az
from scipy import stats

# Configuration
np.random.seed(42)
az.style.use('arviz-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"PyMC version: {pm.__version__}")
print(f"ArviZ version: {az.__version__}")
print("Environment ready!")

## 2. The Pooling Problem: A Motivating Example

**Scenario:** We want to forecast prices for multiple related commodities (e.g., crude oil, heating oil, gasoline).

**Challenge:** Some commodities have limited historical data.

**Question:** Should we:
1. **Pool all data** together (complete pooling) → assumes all commodities identical
2. **Analyze separately** (no pooling) → ignores relationships between commodities
3. **Partially pool** (hierarchical) → borrows strength while respecting differences

### The Three Approaches

**Complete Pooling:**
$$y_{ij} = \alpha + \beta x_{ij} + \epsilon_{ij}$$
- Single $\alpha, \beta$ for all commodities $j$
- **Pro:** Uses all data, stable estimates
- **Con:** Ignores commodity-specific effects

**No Pooling:**
$$y_{ij} = \alpha_j + \beta_j x_{ij} + \epsilon_{ij}$$
- Separate $\alpha_j, \beta_j$ for each commodity
- **Pro:** Flexible, captures differences
- **Con:** Unstable with small samples, doesn't share information

**Partial Pooling (Hierarchical):**
$$\begin{align}
y_{ij} &= \alpha_j + \beta_j x_{ij} + \epsilon_{ij} \\
\alpha_j &\sim \mathcal{N}(\mu_\alpha, \sigma_\alpha^2) \\
\beta_j &\sim \mathcal{N}(\mu_\beta, \sigma_\beta^2)
\end{align}$$
- Commodity-specific parameters drawn from common distributions
- **Pro:** Borrows strength, stable yet flexible
- **Con:** More complex, requires MCMC

## 3. Simulated Multi-Commodity Data

Let's create synthetic data for 8 commodities with varying sample sizes.

In [ ]:
# Simulation setup
np.random.seed(42)

# Global parameters (population-level)
mu_alpha = 80.0  # Mean intercept across commodities
sigma_alpha = 10.0  # Variation in intercepts
mu_beta = -2.0  # Mean slope (storage effect)
sigma_beta = 0.5  # Variation in slopes
sigma_y = 5.0  # Observation noise

# Commodities with varying sample sizes
commodities = ['Crude_Oil', 'Nat_Gas', 'Heating_Oil', 'Gasoline', 
               'Diesel', 'Jet_Fuel', 'Propane', 'Ethanol']
n_commodities = len(commodities)

# Sample sizes (some commodities have little data)
n_obs_per_commodity = [50, 45, 40, 35, 25, 20, 15, 10]

# Generate true parameters for each commodity
true_alpha = np.random.normal(mu_alpha, sigma_alpha, n_commodities)
true_beta = np.random.normal(mu_beta, sigma_beta, n_commodities)

print("True commodity-specific parameters:")
for i, comm in enumerate(commodities):
    print(f"{comm:12s}: α={true_alpha[i]:6.2f}, β={true_beta[i]:6.2f}, n={n_obs_per_commodity[i]}")

In [ ]:
# Generate observations
data_list = []

for j in range(n_commodities):
    n_j = n_obs_per_commodity[j]
    
    # Predictor: storage levels (standardized)
    storage = np.random.normal(0, 1, n_j)
    
    # Response: price
    price = true_alpha[j] + true_beta[j] * storage + np.random.normal(0, sigma_y, n_j)
    
    # Store
    for i in range(n_j):
        data_list.append({
            'commodity': commodities[j],
            'commodity_idx': j,
            'storage': storage[i],
            'price': price[i]
        })

# Create DataFrame
df = pd.DataFrame(data_list)

print(f"\nGenerated {len(df)} total observations across {n_commodities} commodities")
print("\nSample data:")
print(df.head(10))

In [ ]:
# Visualize the data
fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharex=True, sharey=True)
axes = axes.flatten()

for j, comm in enumerate(commodities):
    df_j = df[df['commodity'] == comm]
    
    axes[j].scatter(df_j['storage'], df_j['price'], alpha=0.6, s=50)
    
    # True line
    x_line = np.linspace(-3, 3, 100)
    y_line = true_alpha[j] + true_beta[j] * x_line
    axes[j].plot(x_line, y_line, 'r--', linewidth=2, alpha=0.7, label='True')
    
    axes[j].set_title(f'{comm} (n={n_obs_per_commodity[j]})', fontsize=11)
    axes[j].grid(True, alpha=0.3)
    if j >= 4:
        axes[j].set_xlabel('Storage', fontsize=10)
    if j % 4 == 0:
        axes[j].set_ylabel('Price', fontsize=10)

plt.tight_layout()
plt.show()

**Observation:** Notice how commodities with small sample sizes (Propane, Ethanol) have sparse, noisy data.

## 4. Approach 1: Complete Pooling

Fit a single model ignoring commodity identity.

In [ ]:
# Complete pooling: single regression
with pm.Model() as model_complete:
    # Single set of parameters
    alpha = pm.Normal('alpha', mu=80, sigma=20)
    beta = pm.Normal('beta', mu=0, sigma=5)
    sigma = pm.HalfNormal('sigma', sigma=10)
    
    # Likelihood
    mu = alpha + beta * df['storage'].values
    y_obs = pm.Normal('y_obs', mu=mu, sigma=sigma, observed=df['price'].values)
    
    # Sample
    trace_complete = pm.sample(
        draws=1000, tune=1000, chains=2, cores=1, 
        random_seed=42, return_inferencedata=True
    )

print("\n✅ Complete pooling model fitted")

In [ ]:
# Extract estimates
summary_complete = az.summary(trace_complete, var_names=['alpha', 'beta'])
print("Complete Pooling Estimates:")
print("="*60)
print(summary_complete)
print("="*60)
print(f"\nTrue population means: μ_α={mu_alpha}, μ_β={mu_beta}")

alpha_complete = trace_complete.posterior['alpha'].mean().values
beta_complete = trace_complete.posterior['beta'].mean().values

## 5. Approach 2: No Pooling

Fit separate regressions for each commodity.

In [ ]:
# No pooling: separate models
# For efficiency, we'll use a vectorized approach in PyMC

# Prepare data as ragged arrays
commodity_idx = df['commodity_idx'].values

with pm.Model() as model_no_pooling:
    # Separate parameters for each commodity
    alpha_j = pm.Normal('alpha_j', mu=80, sigma=20, shape=n_commodities)
    beta_j = pm.Normal('beta_j', mu=0, sigma=5, shape=n_commodities)
    sigma = pm.HalfNormal('sigma', sigma=10)
    
    # Likelihood (use indexing)
    mu = alpha_j[commodity_idx] + beta_j[commodity_idx] * df['storage'].values
    y_obs = pm.Normal('y_obs', mu=mu, sigma=sigma, observed=df['price'].values)
    
    # Sample
    trace_no_pooling = pm.sample(
        draws=1000, tune=1000, chains=2, cores=1,
        random_seed=42, return_inferencedata=True
    )

print("\n✅ No pooling model fitted")

In [ ]:
# Extract estimates
alpha_no_pool = trace_no_pooling.posterior['alpha_j'].mean(dim=['chain', 'draw']).values
beta_no_pool = trace_no_pooling.posterior['beta_j'].mean(dim=['chain', 'draw']).values

print("No Pooling Estimates:")
print("="*70)
for j, comm in enumerate(commodities):
    print(f"{comm:12s}: α={alpha_no_pool[j]:6.2f} (true: {true_alpha[j]:6.2f}), "
          f"β={beta_no_pool[j]:6.2f} (true: {true_beta[j]:6.2f})")
print("="*70)

**Observation:** Estimates for low-sample commodities (Propane, Ethanol) are noisy and far from truth.

## 6. Approach 3: Partial Pooling (Hierarchical Model)

Now the key innovation: commodity-specific parameters are drawn from common distributions.

In [ ]:
# Partial pooling: hierarchical model
with pm.Model() as model_partial:
    
    # Hyperparameters (population-level)
    mu_alpha = pm.Normal('mu_alpha', mu=80, sigma=20)
    sigma_alpha = pm.HalfNormal('sigma_alpha', sigma=10)
    
    mu_beta = pm.Normal('mu_beta', mu=0, sigma=5)
    sigma_beta = pm.HalfNormal('sigma_beta', sigma=2)
    
    # Commodity-specific parameters (drawn from population distributions)
    alpha_j = pm.Normal('alpha_j', mu=mu_alpha, sigma=sigma_alpha, shape=n_commodities)
    beta_j = pm.Normal('beta_j', mu=mu_beta, sigma=sigma_beta, shape=n_commodities)
    
    # Observation noise
    sigma = pm.HalfNormal('sigma', sigma=10)
    
    # Likelihood
    mu = alpha_j[commodity_idx] + beta_j[commodity_idx] * df['storage'].values
    y_obs = pm.Normal('y_obs', mu=mu, sigma=sigma, observed=df['price'].values)
    
    # Sample
    trace_partial = pm.sample(
        draws=1000, tune=1000, chains=2, cores=1,
        random_seed=42, return_inferencedata=True
    )

print("\n✅ Partial pooling (hierarchical) model fitted")

In [ ]:
# Extract estimates
alpha_partial = trace_partial.posterior['alpha_j'].mean(dim=['chain', 'draw']).values
beta_partial = trace_partial.posterior['beta_j'].mean(dim=['chain', 'draw']).values

# Population-level estimates
mu_alpha_est = trace_partial.posterior['mu_alpha'].mean().values
mu_beta_est = trace_partial.posterior['mu_beta'].mean().values

print("Partial Pooling Estimates:")
print("="*70)
print(f"Population means: μ_α={mu_alpha_est:.2f} (true: {mu_alpha}), "
      f"μ_β={mu_beta_est:.2f} (true: {mu_beta})\n")
for j, comm in enumerate(commodities):
    print(f"{comm:12s}: α={alpha_partial[j]:6.2f} (true: {true_alpha[j]:6.2f}), "
          f"β={beta_partial[j]:6.2f} (true: {true_beta[j]:6.2f})")
print("="*70)

## 7. Compare All Three Approaches

Let's visualize how estimates differ across pooling strategies.

In [ ]:
# Create comparison DataFrame
comparison = pd.DataFrame({
    'Commodity': commodities,
    'n': n_obs_per_commodity,
    'True_α': true_alpha,
    'Complete_α': [alpha_complete] * n_commodities,
    'NoPoo_α': alpha_no_pool,
    'Partial_α': alpha_partial,
    'True_β': true_beta,
    'Complete_β': [beta_complete] * n_commodities,
    'NoPool_β': beta_no_pool,
    'Partial_β': beta_partial
})

# Compute errors
comparison['Error_Complete_β'] = np.abs(comparison['Complete_β'] - comparison['True_β'])
comparison['Error_NoPool_β'] = np.abs(comparison['NoPool_β'] - comparison['True_β'])
comparison['Error_Partial_β'] = np.abs(comparison['Partial_β'] - comparison['True_β'])

print("Comparison of β estimates:")
print(comparison[['Commodity', 'n', 'True_β', 'Complete_β', 'NoPool_β', 'Partial_β']])

In [ ]:
# Visualize: β estimates vs true values
fig, ax = plt.subplots(figsize=(14, 7))

x_pos = np.arange(n_commodities)
width = 0.2

ax.bar(x_pos - 1.5*width, comparison['True_β'], width, 
       label='True', color='black', alpha=0.8)
ax.bar(x_pos - 0.5*width, comparison['Complete_β'], width, 
       label='Complete Pooling', color='blue', alpha=0.7)
ax.bar(x_pos + 0.5*width, comparison['NoPool_β'], width, 
       label='No Pooling', color='red', alpha=0.7)
ax.bar(x_pos + 1.5*width, comparison['Partial_β'], width, 
       label='Partial Pooling', color='green', alpha=0.7)

ax.set_xlabel('Commodity', fontsize=12)
ax.set_ylabel('β (Storage Effect)', fontsize=12)
ax.set_title('Comparison of β Estimates Across Pooling Strategies', fontsize=14)
ax.set_xticks(x_pos)
ax.set_xticklabels(commodities, rotation=45, ha='right')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

**Key Insights:**

1. **Complete pooling (blue)**: Identical for all commodities, misses variation
2. **No pooling (red)**: Tracks truth well for large samples, erratic for small samples
3. **Partial pooling (green)**: Best of both - respects differences but stabilizes small-sample estimates

## 8. Shrinkage Visualization

**Shrinkage** is the phenomenon where partial pooling "pulls" extreme estimates toward the population mean.

This is especially strong for small-sample commodities.

In [ ]:
# Plot shrinkage for β
fig, ax = plt.subplots(figsize=(12, 8))

# Plot arrows from no-pooling to partial-pooling
for j in range(n_commodities):
    # Arrow from no-pooling to partial-pooling
    ax.annotate('', xy=(n_obs_per_commodity[j], beta_partial[j]),
                xytext=(n_obs_per_commodity[j], beta_no_pool[j]),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='purple', alpha=0.6))
    
    # No pooling estimates (starting points)
    ax.scatter(n_obs_per_commodity[j], beta_no_pool[j], 
               s=80, color='red', alpha=0.7, zorder=3)
    
    # Partial pooling estimates (end points)
    ax.scatter(n_obs_per_commodity[j], beta_partial[j], 
               s=80, color='green', alpha=0.9, zorder=3)
    
    # True values (reference)
    ax.scatter(n_obs_per_commodity[j], true_beta[j], 
               s=60, color='black', marker='x', linewidths=2, zorder=4)

# Population mean
ax.axhline(mu_beta, color='blue', linestyle='--', linewidth=2, 
           label=f'Population Mean: {mu_beta}', alpha=0.7)

# Labels
ax.scatter([], [], s=80, color='red', alpha=0.7, label='No Pooling')
ax.scatter([], [], s=80, color='green', alpha=0.9, label='Partial Pooling')
ax.scatter([], [], s=60, color='black', marker='x', linewidths=2, label='True Value')
ax.plot([], [], color='purple', linewidth=1.5, label='Shrinkage Direction')

ax.set_xlabel('Sample Size (n)', fontsize=12)
ax.set_ylabel('β (Storage Effect)', fontsize=12)
ax.set_title('Shrinkage in Hierarchical Models: Pulling Estimates Toward Population Mean', 
             fontsize=14)
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**What we see:**

- **Small samples** (left side): Large shrinkage (long arrows) - partial pooling doesn't trust noisy estimates
- **Large samples** (right side): Small shrinkage (short arrows) - data is trustworthy
- **Shrinkage direction**: Toward population mean (blue dashed line)
- **Result**: Partial pooling often gets closer to truth (black X) than no pooling

## 9. Quantify Performance: Mean Squared Error

Compare prediction accuracy across all three approaches.

In [ ]:
# Compute MSE for β estimates
mse_complete = np.mean(comparison['Error_Complete_β']**2)
mse_no_pool = np.mean(comparison['Error_NoPool_β']**2)
mse_partial = np.mean(comparison['Error_Partial_β']**2)

print("Mean Squared Error (MSE) for β estimates:")
print("="*50)
print(f"Complete Pooling:  {mse_complete:.4f}")
print(f"No Pooling:        {mse_no_pool:.4f}")
print(f"Partial Pooling:   {mse_partial:.4f}")
print("="*50)
print(f"\nPartial pooling reduces MSE by:")
print(f"  vs Complete: {(1 - mse_partial/mse_complete)*100:.1f}%")
print(f"  vs No Pool:  {(1 - mse_partial/mse_no_pool)*100:.1f}%")

In [ ]:
# MSE by sample size category
comparison['Sample_Category'] = pd.cut(comparison['n'], bins=[0, 20, 35, 100], 
                                        labels=['Small (≤20)', 'Medium (21-35)', 'Large (>35)'])

mse_by_size = comparison.groupby('Sample_Category').agg({
    'Error_Complete_β': lambda x: np.mean(x**2),
    'Error_NoPool_β': lambda x: np.mean(x**2),
    'Error_Partial_β': lambda x: np.mean(x**2)
})

print("\nMSE by Sample Size Category:")
print(mse_by_size)

**Key Finding:** Partial pooling provides the biggest benefit for **small-sample commodities**.

## 10. Forecasting with Hierarchical Models

Generate forecasts for a new, previously unseen commodity.

**Key advantage:** We can use the population distribution to make predictions even without commodity-specific data!

In [ ]:
# Forecast for a new commodity
# We observe its storage level but have no historical price data

storage_new = np.array([1.0])  # New commodity has high storage

# Option 1: Use population-level parameters (no commodity-specific info)
mu_alpha_samples = trace_partial.posterior['mu_alpha'].values.flatten()
mu_beta_samples = trace_partial.posterior['mu_beta'].values.flatten()
sigma_samples = trace_partial.posterior['sigma'].values.flatten()

# Generate forecast using population parameters
price_forecast_population = (
    mu_alpha_samples + mu_beta_samples * storage_new[0] + 
    np.random.normal(0, sigma_samples)
)

print("Forecast for new commodity (using population parameters):")
print(f"  Mean: {price_forecast_population.mean():.2f}")
print(f"  95% PI: [{np.percentile(price_forecast_population, 2.5):.2f}, "
      f"{np.percentile(price_forecast_population, 97.5):.2f}]")

In [ ]:
# Visualize forecast distribution
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(price_forecast_population, bins=50, density=True, alpha=0.7, 
        edgecolor='black', color='green')
ax.axvline(price_forecast_population.mean(), color='red', linestyle='--', 
           linewidth=2, label='Mean Forecast')
ax.axvline(np.percentile(price_forecast_population, 2.5), color='blue', 
           linestyle=':', linewidth=2, label='95% PI')
ax.axvline(np.percentile(price_forecast_population, 97.5), color='blue', 
           linestyle=':', linewidth=2)

ax.set_xlabel('Predicted Price', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Forecast for New Commodity (Zero Historical Data)', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Amazing property:** Hierarchical models let us forecast for **new groups** by using the population distribution!

---

## Exercise 1: Effect of Hyperprior Choice

**Task:** Investigate sensitivity to hyperprior specification.

1. Refit the partial pooling model with tighter hyperpriors:
   - `sigma_alpha = HalfNormal(sigma=2)` instead of 10
   - `sigma_beta = HalfNormal(sigma=0.3)` instead of 2
2. Compare commodity-specific estimates
3. How does stronger shrinkage affect small vs large sample commodities?

In [ ]:
# YOUR CODE HERE

# Fit model with tighter hyperpriors
# Compare estimates with original partial pooling model

In [ ]:
# AUTO-GRADED TEST
def test_exercise_1():
    """Verify tight hyperprior model was fitted."""
    # Test would check for existence of new trace
    print("✅ Exercise 1: Compare shrinkage under different hyperpriors")

# test_exercise_1()  # Uncomment after completing

---

## Exercise 2: Cross-Validation

**Task:** Use leave-one-commodity-out cross-validation to evaluate pooling strategies.

1. For each commodity:
   - Hold it out
   - Fit model on remaining 7 commodities
   - Predict prices for held-out commodity
   - Compute prediction error
2. Compare average prediction error across pooling strategies

**Expected result:** Partial pooling should win, especially for small-sample commodities.

In [ ]:
# YOUR CODE HERE

# Implement leave-one-commodity-out CV
# For simplicity, you can use just one pooling strategy first

---

## Exercise 3: Multiple Predictors

**Task:** Extend to multiple predictors with hierarchical coefficients.

Add a second predictor (e.g., production) to the model:
$$y_{ij} = \alpha_j + \beta_{1j} \times \text{Storage}_{ij} + \beta_{2j} \times \text{Production}_{ij} + \epsilon_{ij}$$

Make both $\beta_{1j}$ and $\beta_{2j}$ hierarchical.

**Challenge:** Should the two coefficients be correlated? (Advanced: use multivariate normal prior)

In [ ]:
# YOUR CODE HERE

# Generate production data
# Extend hierarchical model to include second predictor

---

## Exercise 4: Varying Slopes and Intercepts

**Task:** So far, both intercept and slope vary by commodity. Try two constrained versions:

1. **Varying intercepts only**: All commodities share the same $\beta$
2. **Varying slopes only**: All commodities share the same $\alpha$

Use WAIC or LOO to compare which structure fits best.

In [ ]:
# YOUR CODE HERE

# Fit varying-intercepts-only model
# Fit varying-slopes-only model
# Compare with az.compare()

---

## Solutions

### Exercise 1 Solution (Sketch)

In [ ]:
# SOLUTION
# with pm.Model() as model_tight:
#     mu_alpha = pm.Normal('mu_alpha', mu=80, sigma=20)
#     sigma_alpha = pm.HalfNormal('sigma_alpha', sigma=2)  # Tighter!
#     mu_beta = pm.Normal('mu_beta', mu=0, sigma=5)
#     sigma_beta = pm.HalfNormal('sigma_beta', sigma=0.3)  # Tighter!
#     ...
#
# Result: Estimates shrink more toward population mean
# Small-sample commodities shrink a lot, large-sample less affected

---

## Summary

### Key Takeaways

1. **Complete pooling**: Ignores group differences, stable but biased
2. **No pooling**: Flexible but unstable for small samples
3. **Partial pooling (hierarchical)**: Best of both worlds via shrinkage
4. **Shrinkage is adaptive**: Strong for small samples, weak for large samples
5. **Hierarchical models handle new groups**: Use population distribution for forecasting
6. **Bias-variance tradeoff**: Partial pooling reduces variance at cost of small bias

### When to Use Hierarchical Models

**Ideal scenarios:**
- Multiple related groups (commodities, markets, assets)
- Imbalanced sample sizes across groups
- Need to forecast for new groups
- Want to share information while respecting differences

**Not needed when:**
- Only one group
- Groups are truly independent (no shared structure)
- All groups have large samples

### What's Next

Next notebook: **Energy Complex Hierarchical Model** - Apply to crude oil, nat gas, gasoline simultaneously.

### Additional Resources

- Gelman & Hill (2007): *Data Analysis Using Regression and Multilevel/Hierarchical Models*
- McElreath (2020): *Statistical Rethinking*, Chapter 13
- Efron & Morris (1977): "Stein's Paradox in Statistics" (classic on shrinkage)

---

*Remember: Hierarchical models are about borrowing strength. When you have little data for one group, use what you know about related groups to make better inferences.*